# Versión 2 completa (segundo retorno) — modelo perinatal NCHS

Etapas del framework: **E7** (predictores), puertas **P2** y **P3**, corrida preliminar (**E8**), evaluación formal (**E8–E9**) y puerta **P4**. Versión en inglés: `09_model_v2b.ipynb` (mismo código).

**Decisión del equipo de modelado (23/11/2026).** El segundo retorno consistió en agregar variables prenatales. En la versión 2 (notebook 07) se agregaron cuatro condiciones del embarazo, pero quedaron por fuera otras variables prenatales que tienen los dos archivos. Esta versión completa el segundo retorno con todas ellas: **talla de la madre, cigarrillos en el primer trimestre, medicamentos para la fertilidad y reproducción asistida** (los dos tipos de tratamiento de infertilidad), **sexo del producto** y **mes de la última menstruación** (estación, como seno y coseno). La **edad del padre** se usa solo como análisis de sensibilidad, porque es desconocida con mucha más frecuencia en los reportes de muerte fetal. Las variables se agregaron después de ver los resultados de la versión 2, y así queda registrado.

**Igual que en la versión 2:** misma población (22 semanas o más, todos los registros), imputación por la mediana dentro del entrenamiento, muestra de casos y controles, búsqueda aleatoria de 20 combinaciones por algoritmo, regla de selección del algoritmo principal, validación cruzada 3 × 5 recalibrada, prueba temporal 2022–2023 y puerta P4 del framework 1.1.

**Requisito:** reconstruir antes el adaptador con las variables nuevas: notebook 01 con `REBUILD = True` (unos 2 minutos por año). Necesita también la tabla contrato de la versión 2 (notebook 07). Tiempo estimado: 1 a 2 horas. Salida: `data/outputs_09_model_v2b.txt`.

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
INTERIM_DIR = Path(os.environ.get("NCHS_INTERIM_DIR", ROOT / "data" / "interim"))
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT4_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v2b"))
OUT_FILE = ROOT / "data" / "outputs_09_model_v2b.txt"
SEED = 20261121
YEARS, DEV_YEARS, TEST_YEARS = list(range(2016, 2024)), list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

def coded(s, unknown):
    s = s.astype("Float64").astype(float)
    return s.where(~s.isin(unknown))

def yn(s):
    s = s.astype("string")
    y = s.eq("Y").fillna(False).to_numpy(bool); n = s.eq("N").fillna(False).to_numpy(bool)
    return pd.Series(np.where(y, 1.0, np.where(n, 0.0, np.nan)), index=s.index)

V1_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v1"))
NEW = ["gestational_diabetes", "gestational_hypertension", "eclampsia", "smoked_during_pregnancy"]
V2_DIR = Path(os.environ.get("NCHS_CONTRACT3_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v2"))
ADDED = ["mother_height", "cigarettes_first_trimester", "fertility_drugs", "assisted_reproduction",
         "male_product", "lmp_month_sin", "lmp_month_cos"]


## 1. Tabla de contrato de la versión 2 completa

Toma la tabla de la versión 2 y le une las variables nuevas por `unit_id`. Escribe `data/contract/perinatal_nchs_v2b/year_<año>.parquet` (no se versiona) y muestra el porcentaje de desconocidos de cada variable nueva por componente.

In [ ]:
# Contract table of version 2, completed: the version 2 table (notebook 07) plus the remaining antenatal fields
# shared by the birth and fetal death files, joined by unit_id from the rebuilt adapter (notebook 01):
# mother's height, cigarettes in the 1st trimester, fertility-enhancing drugs and assisted reproduction (the two
# types of infertility treatment), sex of the product and month of the last menses (season, as sine and cosine).
# Father's age is kept in the table for a sensitivity analysis only (unknown far more often on fetal death reports).
REBUILD = False
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)

def sub_yn(s):
    """Y -> 1, N or X (no infertility treatment) -> 0, U or blank -> NaN."""
    s = s.astype("string")
    return pd.Series(np.where(s.eq("Y").fillna(False), 1.0, np.where(s.isin(["N", "X"]).fillna(False), 0.0, np.nan)),
                     index=s.index)

for y in YEARS:
    final = CONTRACT_DIR / f"year_{y}.parquet"
    if final.exists() and not REBUILD:
        continue
    c = pd.read_parquet(V2_DIR / f"year_{y}.parquet")
    d = pd.read_parquet(INTERIM_DIR / f"perinatal_{y}.parquet",
                        columns=["unit_id", "MHTR", "CIG_1", "RF_DRG", "RF_ART", "SEX", "DLMP_MM", "FAGECOMB"])
    add = pd.DataFrame({"unit_id": d["unit_id"]})
    h = coded(d["MHTR"], [99]); add["mother_height"] = h.where(h.between(48, 78))
    add["cigarettes_first_trimester"] = coded(d["CIG_1"], [99]).clip(upper=40)
    add["fertility_drugs"] = sub_yn(d["RF_DRG"]); add["assisted_reproduction"] = sub_yn(d["RF_ART"])
    sx = d["SEX"].astype("string")
    add["male_product"] = np.where(sx.eq("M").fillna(False), 1.0, np.where(sx.eq("F").fillna(False), 0.0, np.nan))
    m = coded(d["DLMP_MM"], [99]); m = m.where(m.between(1, 12))
    add["lmp_month_sin"] = np.sin(2 * np.pi * m / 12); add["lmp_month_cos"] = np.cos(2 * np.pi * m / 12)
    fa = coded(d["FAGECOMB"], [99]); add["father_age"] = fa.where(fa.between(9, 98))
    add[ADDED + ["father_age"]] = add[ADDED + ["father_age"]].astype("float32")
    c = c.merge(add, on="unit_id", how="left", validate="one_to_one")
    assert c["male_product"].notna().mean() > 0.9, f"join failed in {y}: rebuild the adapter (notebook 01, REBUILD = True)"
    c["n_unknown"] = (c["n_unknown"] + c[ADDED].isna().sum(axis=1)).astype("int8")
    tmp = final.with_suffix(".parquet.tmp"); c.to_parquet(tmp, index=False); tmp.replace(final)
    print(f"year {y}: {len(c)} products")
    del c, d, add

DROP = ["unit_id", "group_id", "period", "component", "ga_weeks", "label_main", "label_fetal",
        "label_early_neonatal", "estofd", "label_ge28", "n_unknown", "father_age"]
PREDICTORS = [c for c in pd.read_parquet(CONTRACT_DIR / "year_2023.parquet").columns if c not in DROP]
rows = []
for y in YEARS:
    c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=ADDED + ["father_age", "component", "label_main"])
    fd = c["component"].astype("string").eq("fetal_death").to_numpy()
    for v in ADDED + ["father_age"]:
        rows.append({"year": y, "variable": v, "% unknown, live births": 100 * c.loc[~fd, v].isna().mean(),
                     "% unknown, fetal deaths": 100 * c.loc[fd, v].isna().mean()})
miss = pd.DataFrame(rows).groupby("variable")[["% unknown, live births", "% unknown, fetal deaths"]].mean().round(1)
show(f"Version 2 completed: {len(PREDICTORS)} parameters; unknowns of the added variables by component (mean of 2016-2023)", miss)

## 2. Muestras, puerta P2 y puerta P3

In [ ]:
# Development and temporal-test samples of the MAIN analysis (complete records): every event and
# FRACTION of non-events, with weights 1/FRACTION for non-events.
EXTRA = ["label_main", "label_fetal", "label_early_neonatal", "ga_weeks", "estofd", "n_unknown", "father_age"]
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:
        c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS + EXTRA)
        n += len(c); e += int(c["label_main"].sum())
        parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
P = len(PREDICTORS)
show("Data (22 weeks or more, all records; median imputation inside training, as versions 1 and 2)", pd.DataFrame(
    {"products": [n_dev, n_test], "events": [e_dev, e_test], "sample rows": [len(dev), len(test)]},
    index=["development 2016-2021", "temporal test 2022-2023"]))

# Gate G2: no delivery, product or outcome field among the predictors; no missingness indicators.
# Second return completed (23/11/2026, decision of the modeling team): the remaining antenatal fields
# shared by both files enter; father's age is excluded from the main model (sensitivity only).
EXCLUDED = ["ga_weeks", "birthweight", "estofd", "component", "label_main", "n_unknown"]
show("Gate G2: design checks", pd.Series({
    "no delivery, product or outcome field among the predictors": not set(PREDICTORS) & set(EXCLUDED),
    "no missingness indicator among the predictors": not any("missing" in p or "unknown" in p for p in PREDICTORS),
    "pregnancy conditions included by decision of the modeling team (D4 revised)": all(v in PREDICTORS for v in NEW),
    "added antenatal fields included; father's age excluded from the main model": all(v in PREDICTORS for v in ADDED) and "father_age" not in PREDICTORS}))

# Gate G3: Riley criteria with the observed Cox-Snell R2 of a cross-validated logistic regression
X_s, y_s, w_s = dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy()
oof = np.zeros(len(dev))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_s, y_s):
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1e6, max_iter=2000))
    m.fit(X_s.iloc[tr], y_s[tr], logisticregression__sample_weight=w_s[tr]); oof[te] = m.predict_proba(X_s.iloc[te])[:, 1]
p_ = np.clip(oof, 1e-12, 1 - 1e-12)
L1 = float(np.sum(w_s * (y_s * np.log(p_) + (~y_s) * np.log(1 - p_)))); n_w = float(w_s.sum()); phi = e_dev / n_dev
L0 = n_w * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
R2 = 1 - np.exp(2 * (L0 - L1) / n_w); R2max = 1 - np.exp(2 * L0 / n_w)
n1 = P / ((0.9 - 1) * np.log(1 - R2 / 0.9)); S2 = R2 / (R2 + 0.05 * R2max); n2 = P / ((S2 - 1) * np.log(1 - R2 / S2))
n_req = int(np.ceil(max(n1, n2, (1.96 / 0.05) ** 2 * phi * (1 - phi))))
show("Gate G3: sample size (Riley et al.)", pd.Series({
    "parameters": P, "events per parameter": round(e_dev / P, 1), "Cox-Snell R2 observed": round(R2, 5),
    "% of maximum": round(100 * R2 / R2max, 1), "products required": n_req, "products available": n_dev,
    "gate G3": "PASSED" if n_dev >= n_req else "NOT PASSED"}))

# Preliminary run (same procedure as notebook 04): 80/20 split, random search with the same budget per learner
X = X_s.astype("float32"); y = y_s; w = w_s
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=SEED)

## 3. Corrida preliminar: búsqueda aleatoria y selección del algoritmo principal

In [ ]:
# Hyperparameter search (correction of 2026-11-22: the first runs tried only 2-3 values of one
# hyperparameter per learner). Random search with the SAME budget for every learner: N_CONFIGS
# combinations drawn from prespecified ranges, scored by AUROC in 3 inner folds of a stratified 30%
# subsample of the fit part; the best combination is then refitted on the whole fit part.
# Every learner imputes the median inside training (decision D3). XGBoost: number of trees chosen by
# early stopping (up to 2,000, patience 50) in each inner fold; the refit uses their mean.
from sklearn.base import clone
N_CONFIGS = 3 if QUICK else 20
SEARCH_SHARE = 0.30
rs = np.random.default_rng(SEED)
loguni = lambda lo, hi: float(np.exp(rs.uniform(np.log(lo), np.log(hi))))

def sample_config(name):
    if name == "LR":
        return {"C": loguni(1e-4, 10), "penalty": str(rs.choice(["l1", "l2"]))}
    if name in ("RF", "ET"):
        return {"max_depth": [8, 12, 16, 20, 30, None][rs.integers(6)],
                "min_samples_leaf": int(loguni(20, 1000)),
                "max_features": ["sqrt", 0.3, 0.5][rs.integers(3)]}
    return {"max_depth": int(rs.integers(3, 11)), "learning_rate": loguni(0.01, 0.2),
            "min_child_weight": loguni(1, 100), "subsample": float(rs.uniform(0.6, 1.0)),
            "colsample_bytree": float(rs.uniform(0.6, 1.0)), "reg_lambda": loguni(0.1, 10),
            "reg_alpha": loguni(1e-3, 1)}

def build(name, cfg, n_estimators=None):
    imp = SimpleImputer(strategy="median")
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=n_estimators or (100 if QUICK else 2000), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, eval_metric="auc", **cfg))

def fit_w(est, X_, y_, w_, **extra):
    step = est.steps[-1][0]
    return est.fit(X_, y_, **{f"{step}__sample_weight": w_, **{f"{step}__{k}": v for k, v in extra.items()}})

X_se, _, y_se, _, w_se, _ = train_test_split(X_tr, y_tr, w_tr, train_size=SEARCH_SHARE, stratify=y_tr, random_state=SEED)
inner = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_se, y_se))
fitted, pred, grid_rows, trials = {}, {}, [], []
for name in ["LR", "RF", "ET", "XGB"]:
    t = time.time(); best = (-1, None, None)
    for k in range(N_CONFIGS):
        cfg = sample_config(name); aucs, iters = [], []
        for tr, va in inner:
            est = build(name, cfg)
            if name == "XGB":
                imp = SimpleImputer(strategy="median").fit(X_se.iloc[tr])
                xgb = est.steps[-1][1].set_params(early_stopping_rounds=50)
                xgb.fit(imp.transform(X_se.iloc[tr]), y_se[tr], sample_weight=w_se[tr],
                        eval_set=[(imp.transform(X_se.iloc[va]), y_se[va])], verbose=False)
                p = xgb.predict_proba(imp.transform(X_se.iloc[va]))[:, 1]; iters.append(xgb.best_iteration + 1)
            else:
                p = fit_w(est, X_se.iloc[tr], y_se[tr], w_se[tr]).predict_proba(X_se.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_se[va], p))
        score = float(np.mean(aucs))
        trials.append({"learner": name, "inner AUROC": score, **{k: (round(v, 4) if isinstance(v, float) else str(v)) for k, v in cfg.items()}})
        if score > best[0]:
            best = (score, cfg, int(np.mean(iters)) if iters else None)
    score, cfg, n_iter = best
    fitted[name] = fit_w(build(name, cfg, n_estimators=n_iter), X_tr, y_tr, w_tr)
    pred[name] = fitted[name].predict_proba(X_te)[:, 1]
    shown = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in cfg.items()}
    if n_iter: shown["n_estimators"] = n_iter
    grid_rows.append({"learner": name, "best combination": json.dumps(shown), "inner AUROC": round(score, 4),
                      "minutes": round((time.time() - t) / 60, 1)})
    print(name, "done in", grid_rows[-1]["minutes"], "min")
show(f"Random search ({N_CONFIGS} combinations per learner, 3 inner folds on {int(SEARCH_SHARE * 100)}% of the fit part)",
     pd.DataFrame(grid_rows).set_index("learner"))
show("Spread of the inner AUROC over the combinations tried (how much the search mattered)",
     pd.DataFrame(trials).groupby("learner")["inner AUROC"].agg(["min", "median", "max"]).round(4))
trials_df = pd.DataFrame(trials)
for name in ["LR", "RF", "ET", "XGB"]:
    t_ = trials_df[trials_df["learner"] == name].dropna(axis=1, how="all").drop(columns="learner")
    show(f"All combinations tried: {name}", t_.sort_values("inner AUROC", ascending=False).round(4).reset_index(drop=True))

# Stacked ensemble with a logistic meta-learner (comparison only): out-of-fold predictions of each
# tuned base learner inside the fit part train the meta-learner; the base learners fitted on the
# whole fit part score the comparison part.
t = time.time()
oof = np.zeros((len(X_tr), len(fitted)))
for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_tr, y_tr):
    for j, est in enumerate(fitted.values()):
        m = fit_w(clone(est), X_tr.iloc[tr], y_tr[tr], w_tr[tr])
        oof[va, j] = m.predict_proba(X_tr.iloc[va])[:, 1]
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
meta = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # meta-learner on the logit scale
meta.fit(logit(oof), y_tr, logisticregression__sample_weight=w_tr)
pred["Stack"] = meta.predict_proba(logit(np.column_stack([pred[k] for k in fitted])))[:, 1]
print("Stack done in", round((time.time() - t) / 60, 1), "min")

# Comparison on the 20% part. AUROC does not depend on sampling the non-events; AUPRC and the
# observed/expected ratio use the weights so they refer to the full population.
B = 50 if QUICK else 300
boot = np.random.default_rng(SEED)
idx_pos, idx_neg = np.where(y_te)[0], np.where(~y_te)[0]
boot_idx = [np.concatenate([boot.choice(idx_pos, len(idx_pos)), boot.choice(idx_neg, len(idx_neg))]) for _ in range(B)]

def auroc(p, ix=None):
    ix = slice(None) if ix is None else ix
    return roc_auc_score(y_te[ix], p[ix])

rows = {}
for name, p in pred.items():
    a = [auroc(p, ix) for ix in boot_idx]
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    cut = p[o][np.searchsorted(cw, 0.90 * cw[-1])]          # weighted 90th percentile of risk
    top = p >= cut
    rows[name] = {"AUROC": round(auroc(p), 4), "AUROC 2.5%": round(np.percentile(a, 2.5), 4),
                  "AUROC 97.5%": round(np.percentile(a, 97.5), 4),
                  "AUPRC (weighted)": round(average_precision_score(y_te, p, sample_weight=w_te), 4),
                  "observed/expected": round(float(y_te.sum() / np.sum(w_te * p)), 3),
                  "events in top 10% (%)": round(100 * float(y_te[top].sum() / y_te.sum()), 1)}
res = pd.DataFrame(rows).T
show(f"Preliminary comparison on the 20% part (prevalence {y_te.sum() / w_te.sum():.5f})", res)

# Prespecified selection rule (outcome card, S8): primary = best base learner by AUROC; when the
# difference with a simpler learner is under 0.005, the simpler one is preferred (LR < XGB < RF/ET).
order = ["LR", "XGB", "RF", "ET"]
best = max(order, key=lambda k: res.loc[k, "AUROC"])
primary = next(k for k in order if res.loc[best, "AUROC"] - res.loc[k, "AUROC"] < 0.005)
d_stack = [auroc(pred["Stack"], ix) - auroc(pred[primary], ix) for ix in boot_idx]
lo, hi = np.percentile(d_stack, [2.5, 97.5])
adopt = (res.loc["Stack", "AUROC"] - res.loc[primary, "AUROC"] >= 0.005) and lo > 0
show("Selection", pd.Series({"best base learner by AUROC": best, "primary learner (rule)": primary,
                             "Stack - primary AUROC": f"{np.mean(d_stack):+.4f} ({lo:+.4f} to {hi:+.4f})",
                             "ensemble adopted": adopt}))
BEST = {g['learner']: json.loads(g['best combination']) for g in grid_rows}

## 4. Evaluación formal: validación cruzada repetida y recalibrada

In [ ]:
# Formal evaluation of the primary learner (and logistic regression as reference), as in notebook 05
REPEATS, FOLDS, B = (1, 3, 30) if QUICK else (3, 5, 200)
def make_model(name):
    cfg = dict(BEST[name]); n_iter = cfg.pop("n_estimators", None)
    return build(name, cfg, n_estimators=n_iter)
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn)[:, 1]).reshape(-1, 1))[:, 1]

# Repeated stratified cross-validation (products are independent units; no mother identifier):
# REPEATS x FOLDS, recalibration inside each training part, out-of-fold predictions pooled per repeat.
X_dev, y_dev, w_dev = dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy()
MODELS = list(dict.fromkeys([primary, "LR"]))
oof = {name: np.zeros((REPEATS, len(dev))) for name in MODELS}
for r in range(REPEATS):
    t = time.time()
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED + r).split(X_dev, y_dev):
        for name in oof:
            f = fit_calibrated(name, X_dev.iloc[tr], y_dev[tr], w_dev[tr], SEED + r)
            oof[name][r, te] = f(X_dev.iloc[te])
    print(f"repeat {r + 1} done in {(time.time() - t) / 60:.1f} min")

KEYS = ["AUROC", "AUPRC", "calibration slope", "calibration intercept", "observed/expected"]
rows = {}
for name in oof:
    per_rep = pd.DataFrame([metrics(y_dev, oof[name][r], w_dev) for r in range(REPEATS)])
    lo, hi = boot_ci(y_dev, oof[name][0], w_dev, KEYS, SEED)
    row = per_rep.mean().round(4).to_dict()
    row.update({f"{k} 2.5%": round(lo[k], 4) for k in KEYS}); row.update({f"{k} 97.5%": round(hi[k], 4) for k in KEYS})
    row["AUROC SD across repeats"] = round(per_rep["AUROC"].std(ddof=0), 4)
    rows[name] = row
main = pd.DataFrame(rows)
show(f"Internal validation: {REPEATS} x {FOLDS}-fold cross-validation, recalibrated (development 2016-2021)", main)

## 5. Validación temporal (C4), análisis de sensibilidad (E9), razones de odds de las variables agregadas y sensibilidad con la edad del padre

In [ ]:
# Temporal validation (criterion C4): fit on the whole development sample, recalibrate inside it,
# apply once to 2022-2023.
X_t, y_t, w_t = test[PREDICTORS], test["label_main"].to_numpy(bool), test["w"].to_numpy()
temporal = {}
for name in MODELS:
    f = fit_calibrated(name, X_dev, y_dev, w_dev, SEED)
    p_t = f(X_t)
    m = metrics(y_t, p_t, w_t); lo, hi = boot_ci(y_t, p_t, w_t, KEYS, SEED + 7)
    temporal[name] = {**{k: round(v, 4) for k, v in m.items()},
                      **{f"{k} 2.5%": round(lo[k], 4) for k in KEYS}, **{f"{k} 97.5%": round(hi[k], 4) for k in KEYS}}
    pass
show("Temporal validation: development 2016-2021, single test 2022-2023", pd.DataFrame(temporal))

# Sensitivity analyses of the outcome card (E9), on the pooled out-of-fold predictions of XGBoost (repeat 1)
p0 = oof[primary][0]
complete = (dev["n_unknown"] == 0).to_numpy()
ge28 = (dev["ga_weeks"] >= 28).to_numpy()
is_fd = dev["label_fetal"].to_numpy(bool); is_en = dev["label_early_neonatal"].to_numpy(bool)
antepartum = dev["estofd"].astype("string").eq("N").fillna(False).to_numpy(bool)
no_event = ~y_dev
sens = {
    "all products (main)": (np.ones(len(dev), bool), y_dev),
    "complete records only (D3)": (complete, y_dev),
    "28 weeks or more": (ge28, y_dev),
    "fetal deaths vs no event": (is_fd | no_event, is_fd),
    "antepartum fetal deaths vs no event": (antepartum | no_event, antepartum),
    "early neonatal deaths vs no event": (is_en | no_event, is_en),
}
srows = {}
for k, (mask, yy) in sens.items():
    srows[k] = {"products (weighted)": int(round(w_dev[mask].sum())), "events": int(yy[mask].sum()),
                "AUROC": round(roc_auc_score(yy[mask], p0[mask]), 4)}
show("Sensitivity analyses (primary learner, out-of-fold predictions)", pd.DataFrame(srows).T)

# Diagnostic of the four pregnancy conditions (dependence on the length of pregnancy). If a condition is
# recorded less often when the pregnancy ends early only because there was no time to diagnose it, the
# event rate WITHOUT the condition is higher in the 22-27 week products and the adjusted odds ratio can
# fall below 1 (an apparently protective effect that only reflects the end of pregnancy).
band = pd.cut(dev["ga_weeks"], [21, 27, 36, 48], labels=["22-27", "28-36", "37+"])
drows = []
for v in NEW:
    for b in band.cat.categories:
        m = (band == b).to_numpy()
        for val, lab in ((1.0, "yes"), (0.0, "no")):
            k = m & (dev[v].to_numpy() == val)
            if w_dev[k].sum() > 0:
                drows.append({"variable": v, "weeks at the end": b, "condition": lab,
                              "products (weighted)": int(round(w_dev[k].sum())),
                              "events per 1,000": round(1000 * w_dev[k & y_dev].sum() / w_dev[k].sum(), 1)})
show("Event rate with and without each new condition, by gestational age at the end",
     pd.DataFrame(drows).set_index(["variable", "weeks at the end", "condition"]))
lr = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(C=1e6, max_iter=2000))
orow = {}
for lab, m in [("all products", np.ones(len(dev), bool)), ("28 weeks or more", (dev["ga_weeks"] >= 28).to_numpy())]:
    lr.fit(X_dev[m], y_dev[m], logisticregression__sample_weight=w_dev[m])
    coef = dict(zip(PREDICTORS, lr[-1].coef_[0]))
    orow[lab] = {v: round(float(np.exp(coef[v])), 3) for v in NEW}
show("Adjusted odds ratios of the new conditions (logistic regression with all predictors)", pd.DataFrame(orow))

# Adjusted odds ratios of the added variables (logistic regression with all predictors, whole population)
lr2 = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(C=1e6, max_iter=3000))
lr2.fit(X_dev, y_dev, logisticregression__sample_weight=w_dev)
coef = dict(zip(PREDICTORS, lr2[-1].coef_[0]))
show("Adjusted odds ratios of the added variables (per unit: inch of height, cigarette per day; yes vs no; sine and cosine of the month)",
     pd.Series({v: round(float(np.exp(coef[v])), 3) for v in ADDED}))

# Sensitivity: father's age added (unknown far more often on fetal death reports; median imputation)
Xf = dev[PREDICTORS + ["father_age"]].copy()
pf = np.zeros(len(dev))
for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(Xf, y_dev):
    pf[te] = fit_calibrated(primary, Xf.iloc[tr], y_dev[tr], w_dev[tr], SEED)(Xf.iloc[te])
show("Sensitivity: father's age added to the primary learner", pd.Series({
    "AUROC without father's age (main, repeat 1)": round(roc_auc_score(y_dev, oof[primary][0]), 4),
    "AUROC with father's age": round(roc_auc_score(y_dev, pf), 4)}))

## 6. Puerta P4, matriz de confusión y curva de decisión

In [ ]:
# Gate G4 (classification). C1: with option A the information precedes every event (applies by design).
# C2: every event is dated after the prediction point, so C2 is judged on the AUROC of the main
# population (>= 0.75 with lower 95% limit >= 0.70). C3: not applicable. C4: temporal AUROC >= 0.70
# and temporal calibration slope between 0.90 and 1.10 (framework 1.1, 22/11/2026).
x, tt = main[primary], temporal[primary]
c2 = (x["AUROC"] >= 0.75) and (x["AUROC 2.5%"] >= 0.70)
c4 = (tt["AUROC"] >= 0.70) and (0.90 <= tt["calibration slope"] <= 1.10)   # framework 1.1: tolerance band
top10 = x["events in top 10% (%)"] / 10
if x["AUROC"] >= 0.75 and c2 and c4:
    cls = "PREDICTOR"
elif x["AUROC"] >= 0.75 and top10 >= 2:
    cls = "EARLY IDENTIFIER"
else:
    cls = "NEITHER"
if tt["AUROC"] < 0.65:
    cls = "NEITHER"
show("Gate G4: classification of version 2 completed (primary learner)", pd.Series({
    "C1 events already present": "met by design (antenatal information precedes every event)",
    "C2 outcome with time": f"{'met' if c2 else 'NOT met'} (AUROC {x['AUROC']:.3f}, 95% CI {x['AUROC 2.5%']:.3f}-{x['AUROC 97.5%']:.3f}; required >= 0.75 and >= 0.70)",
    "C3 detection bias": "not applicable (the outcome does not depend on a test)",
    "C4 stability over time": f"{'met' if c4 else 'NOT met'} (temporal AUROC {tt['AUROC']:.3f}; slope {tt['calibration slope']:.2f}, required 0.90-1.10; 95% CI {tt['calibration slope 2.5%']:.2f}-{tt['calibration slope 97.5%']:.2f})",
    "events in top 10% vs chance": f"{top10:.2f} times",
    "classification": cls}))

# Accuracy and confusion matrix of the primary model (requested; not a framework metric)
cm = []
for share in (0.10, 0.05):
    p = oof[primary][0]; o = np.argsort(p); cw = np.cumsum(w_dev[o])
    flag = p >= p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
    tp, fn = w_dev[flag & y_dev].sum(), w_dev[~flag & y_dev].sum()
    fp, tn = w_dev[flag & ~y_dev].sum(), w_dev[~flag & ~y_dev].sum()
    cm.append({"flagged": f"top {int(share * 100)}%", "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
               "accuracy": round((tp + tn) / (tp + tn + fp + fn), 4), "sensitivity": round(tp / (tp + fn), 3),
               "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4), "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrix of the primary learner (out-of-fold, weighted to the development population)", pd.DataFrame(cm).set_index("flagged"))

# Decision curve (net benefit) at a few risk thresholds
dc = []
for thr in (0.005, 0.01, 0.02, 0.05):
    p = oof[primary][0]; flag = p >= thr; n = w_dev.sum()
    nb = w_dev[flag & y_dev].sum() / n - w_dev[flag & ~y_dev].sum() / n * thr / (1 - thr)
    prev = w_dev[y_dev].sum() / n
    dc.append({"threshold": thr, "net benefit model": round(nb, 5), "net benefit treat all": round(prev - (1 - prev) * thr / (1 - thr), 5),
               "% flagged": round(100 * w_dev[flag].sum() / n, 2)})
show("Decision curve (primary learner)", pd.DataFrame(dc).set_index("threshold"))

OUT_FILE.write_text("Version 2 completed (second return): version 2 + remaining shared antenatal fields\n" + "\n".join(lines) + "\n", encoding="utf-8")